# 読めない博物誌 ── 標本をつくる

このノートブックは、図鑑『読めない博物誌』の中身を生成します。

1. **ConceptLab** で「既存のどの生きものにも似ない」生きものを 1 体ずつ生み出す
   - 250 ステップごとに BLIP-2 が画像を見て「これは何か」と答え、その答えが否定語として加わる
   - 生きものはその否定語から遠ざかるように変化する。この **否定語の履歴** が、図鑑で唯一読める説明文になる
2. 各標本の **図版** を生成する
3. 各標本の **座標**（CLIP ViT-L/14 の語彙空間における主成分座標）を書き出し、図鑑が読み込む `specimens.js` にまとめる

**ランタイム**：GPU。本番は L4 か A100 を推奨します。無料枠の T4（15GB）でも動くよう、GPU のメモリが 20GB 未満のときは「これは何か」に答えるモデルを軽量版（BLIP VQA）に切り替えます
**所要時間**：1 体あたり数十分が目安です（GPU によって変わります）。試運転（`SMOKE_TEST = True`）は 2026-10-04 に完走済みなので、既定は本番の設定です。
**保存先**：Google ドライブの `yomenai-hakubutsushi/`。途中で切断されても、できた標本は飛ばして再開します。

使うもの：[ConceptLab](https://github.com/kfirgoldberg/ConceptLab)（MIT）、Kandinsky 2.1、BLIP-2（flan-t5-xl）。作品として発表する前に、各モデルのライセンスを配布元で確認してください。

In [ ]:
#@title 1. 設定
# 部（章）ごとに、ConceptLab に与えるカテゴリ語と体数を決める。
# カテゴリ語は CLIP の語彙にある英単語 1 語にする（例：creature, animal, plant, insect, bird, fish）。
# BLIP-2 には「What kind of <カテゴリ語> is in this photo?」と問われる。
CHAPTERS = [
    {"key": "creature", "title": "第一部　生きもの", "count": 8},
    # {"key": "plant",  "title": "第二部　草木", "count": 6},
    # {"key": "insect", "title": "第三部　虫",   "count": 6},
]

STEPS = 2500          # 論文の設定。短くすると速いが、既存の生きものから離れきらない
HYBRID_STEPS = 2000   # 交配種の学習ステップ数（ConceptLab の既定値）
SMOKE_TEST = False    # True にすると 1 体だけ、300 ステップで試運転する（結果は smoke/ に分けて保存）。2026-10-04 に試運転済み

# 「これは何か」に答えるモデル。"auto" は GPU のメモリで選ぶ（20GB 未満なら軽量版）
#   "blip2"：BLIP-2 flan-t5-xl（論文と同じ・約 8GB）／ "blip"：BLIP VQA base（軽量版・約 1GB）
VLM = "auto"

# 交配種（任意）：(親A の番号, 親B の番号)。番号は 5. で付く No. と同じ
HYBRIDS = [
    # (1, 2),
]

# 図版のプロンプト。{} に生きものの擬似単語が入る。1 番目が図鑑の主図版になる
PLATE_PROMPTS = [
    "a natural history illustration of a {}, antique hand-colored engraving, scientific plate, plain paper background",
    "a photo of a {} in its natural habitat, wildlife photography",
    "a museum specimen of a {} in a glass case, natural history museum",
    "a close-up photo of a {}",
]
PLATE_SEEDS = [0, 1]

# 巻末「手がかり」の頁に載せる既知の語
ROSETTA_WORDS = ["human", "cat", "dog", "bird", "fish", "tree", "flower", "stone", "water", "moon", "hand", "eye"]

OUT = "/content/drive/MyDrive/yomenai-hakubutsushi"
WEIGHTS = "/content/kandinsky2"

In [ ]:
#@title 2. 環境の準備（初回のみ・数分）
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(OUT, exist_ok=True)

!git clone -q https://github.com/kfirgoldberg/ConceptLab.git /content/ConceptLab || true
!git clone -q https://github.com/openai/CLIP.git /content/CLIP || true
!pip -q install pyrallis omegaconf einops pytorch-lightning sentencepiece ftfy regex
%cd /content/ConceptLab
!rm -rf ./clip && cp -r /content/CLIP/clip ./clip

# 2023 年のコードを現在の Colab で動かすための修正
#   - huggingface_hub から削除された cached_download（重みは 3. で先にダウンロードする）
#   - NumPy から削除された np.int
#   - Matplotlib から削除された plt.cm.get_cmap
#   - setuptools に依存する pkg_resources（古い版の CLIP のみ）
!sed -i 's/^from huggingface_hub import hf_hub_url, cached_download$/from huggingface_hub import hf_hub_url\ndef cached_download(*a, **k): raise RuntimeError("weights are downloaded by the notebook")/' kandinsky2/__init__.py kandinsky2/load_utils.py
!sed -i 's/dtype=np\.int)/dtype=int)/' kandinsky2/model/resample.py
!sed -i "s/plt\.cm\.get_cmap('tab20')/matplotlib.colormaps['tab20']/" training/coach.py
!sed -i 's/^from pkg_resources import packaging$/import packaging.version/' clip/clip.py
!grep -c "def cached_download" kandinsky2/__init__.py kandinsky2/load_utils.py

In [ ]:
#@title 3. Kandinsky 2.1 の重みをダウンロード（初回のみ・数 GB）
from huggingface_hub import hf_hub_download

REPO = "ai-forever/Kandinsky_2.1"
FILES = ["decoder_fp16.ckpt", "prior_fp16.ckpt", "movq_final.ckpt", "ViT-L-14_stats.th"] + [
    f"text_encoder/{name}" for name in
    ["config.json", "pytorch_model.bin", "sentencepiece.bpe.model",
     "special_tokens_map.json", "tokenizer.json", "tokenizer_config.json"]
]
for f in FILES:
    hf_hub_download(REPO, f, local_dir=f"{WEIGHTS}/2_1")
print("done")

In [ ]:
#@title 4. モデルを読み込む（ConceptLab が毎回モデルを読み直さないように差し替える）
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import sys, json, time, re, shutil, functools
from pathlib import Path
import torch
import torch.nn as nn
sys.path.insert(0, "/content/ConceptLab")

# PyTorch 2.6 以降の torch.load は既定で weights_only=True。古いチェックポイントを読むために元の挙動へ戻す
# （再実行しても二重に包まないよう、包まれていない元の torch.load までたどってから包む）
import inspect
_original_load = inspect.unwrap(torch.load)
@functools.wraps(_original_load)
def _load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _original_load(*args, **kwargs)
torch.load = _load

from kandinsky2 import load_utils
from training.coach import Coach
from training.coach_evolution import CoachEvolution, TrainEvolutionConfig
from training.train_config import TrainConfig

if "MODEL" not in globals():
    MODEL = load_utils.get_model(Path(WEIGHTS), "cuda:0")
    torch.cuda.empty_cache()
PLACEHOLDER = "mycreature"

# ConceptLab は学習のたびに語彙へ擬似単語を 1 つ足す。標本ごとに元の語彙へ戻すための控え
if "VOCAB" not in globals():
    _tok = MODEL.tokenizer2
    VOCAB = dict(encoder=dict(_tok.encoder), decoder=dict(_tok.decoder), cache=dict(_tok.cache),
                 embedding=MODEL.clip_model.token_embedding)

def restore_vocab():
    MODEL.tokenizer2.encoder = dict(VOCAB["encoder"])
    MODEL.tokenizer2.decoder = dict(VOCAB["decoder"])
    MODEL.tokenizer2.cache = dict(VOCAB["cache"])
    MODEL.clip_model.token_embedding = VOCAB["embedding"]
    MODEL.clip_model.eval()
    MODEL.text_encoder.eval()

def add_token(token, embedding):
    """scripts/infer.py と同じ手順で、学習済みの擬似単語を語彙に足す"""
    tok = MODEL.tokenizer2
    idx = len(tok.encoder)
    tok.encoder[token] = idx
    tok.decoder[idx] = token
    tok.cache[token] = token
    old = MODEL.clip_model.token_embedding.weight.data
    new = nn.Embedding(old.shape[0] + 1, old.shape[1]).to(old.device)
    new.weight.data[:-1] = old.float()
    new.weight.data[-1] = embedding.to(old.device).float()
    MODEL.clip_model.token_embedding = new

# 「これは何か」に答えるモデル。T4（15GB）には Kandinsky（約 7GB）と BLIP-2（約 8GB）が同時に載らないので、軽量版に切り替える
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
USE_BLIP2 = VLM == "blip2" or (VLM == "auto" and GPU_GB >= 20)
VLM_NAME = "BLIP-2 flan-t5-xl" if USE_BLIP2 else "BLIP VQA base"
print(f"GPU：{torch.cuda.get_device_name(0)}（{GPU_GB:.0f} GB）／ 否定語を答えるモデル：{VLM_NAME}")

_BLIP = {}
def _load_blip_once(self):
    if not self.cfg.live_negatives:
        return None, None
    if not _BLIP:
        if USE_BLIP2:
            from transformers import Blip2Processor, Blip2ForConditionalGeneration
            _BLIP["processor"] = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl")
            _BLIP["model"] = Blip2ForConditionalGeneration.from_pretrained(
                "Salesforce/blip2-flan-t5-xl", device_map="auto", dtype=torch.float16)
        else:
            # ConceptLab の query_vlm と同じ呼び方（processor(画像, 質問) → generate → decode）で使える
            from transformers import BlipProcessor, BlipForQuestionAnswering
            _BLIP["processor"] = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
            _BLIP["model"] = BlipForQuestionAnswering.from_pretrained(
                "Salesforce/blip-vqa-base", dtype=torch.float16).to("cuda")
    return _BLIP["processor"], _BLIP["model"]

Coach.get_model = lambda self: MODEL
Coach.load_blip_vlm = _load_blip_once

SPEC_DIR = Path(OUT) / ("smoke" if SMOKE_TEST else "specimens")
SPEC_DIR.mkdir(parents=True, exist_ok=True)
print("保存先:", SPEC_DIR)

In [ ]:
#@title 5. 標本をつくる（ConceptLab）
def specimen_plan():
    plan, sid = [], 1
    for ch in CHAPTERS:
        for k in range(1 if SMOKE_TEST else ch["count"]):
            plan.append({"id": sid, "chapter": ch["key"], "category": ch["key"], "seed": k})
            sid += 1
        if SMOKE_TEST:
            break
    return plan

for spec in specimen_plan():
    d = SPEC_DIR / f"{spec['id']:03d}"
    if (d / "meta.json").exists():
        print(f"No.{spec['id']:03d} はできています。飛ばします")
        continue
    d.mkdir(exist_ok=True)
    t0 = time.time()
    restore_vocab()
    cfg = TrainConfig(
        placeholder_token=PLACEHOLDER,
        initializer_token=spec["category"],
        positive_classes=[spec["category"]],
        live_negatives=True,
        steps=300 if SMOKE_TEST else STEPS,
        log_image_frequency=100 if SMOKE_TEST else 250,
        log_embed_frequency=250,
        seed=spec["seed"],
        cache_root=Path(WEIGHTS),
        output_dir=d / "run",
    )
    coach = Coach(cfg)
    coach.train()
    emb = torch.load(d / "run" / "learned_embeds.bin")["t2"][PLACEHOLDER].float().cpu()
    torch.save(emb, d / "embedding.pt")
    meta = dict(spec, negatives=list(coach.cfg.negative_classes), parents=None, vlm=VLM_NAME,
                minutes=round((time.time() - t0) / 60, 1))
    (d / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=1))
    print(f"No.{spec['id']:03d}：{meta['minutes']} 分／否定語：{meta['negatives']}")
    del coach
    restore_vocab()
    torch.cuda.empty_cache()

In [ ]:
#@title 6. 図版をつくる
def make_plates(d):
    emb = torch.load(d / "embedding.pt")
    out = d / "plates"
    out.mkdir(exist_ok=True)
    restore_vocab()
    add_token(PLACEHOLDER, emb)
    for pi, prompt in enumerate(PLATE_PROMPTS):
        for s in PLATE_SEEDS:
            f = out / f"p{pi}_s{s}.jpg"
            if f.exists():
                continue
            # scripts/infer.py と同じ設定（デコーダ側のテキストは空、prior 側に擬似単語入りの文を渡す）
            img = MODEL.generate_text2img(
                "", num_steps=50, batch_size=1, guidance_scale=7.5, h=512, w=512,
                sampler="p_sampler", prior_cf_scale=4, prior_steps="5",
                img_prompt=prompt.format(PLACEHOLDER), seed=s)[0]
            img.save(f, quality=92)
    restore_vocab()

for d in sorted(p for p in SPEC_DIR.iterdir() if (p / "meta.json").exists()):
    make_plates(d)
    print(d.name, "の図版ができました")

In [ ]:
#@title 7.（任意）交配種をつくる
# 両親の写真調の図版（生息地・接写）から、両者を混ぜた新しい生きものを学習する
for a, b in HYBRIDS:
    d = SPEC_DIR / f"h{a:03d}_{b:03d}"
    if (d / "meta.json").exists():
        print(d.name, "はできています。飛ばします")
        continue
    parent_dirs = []
    for pid in (a, b):
        src = SPEC_DIR / f"{pid:03d}" / "plates"
        dst = d / "parents" / f"{pid:03d}"
        dst.mkdir(parents=True, exist_ok=True)
        for name in ["p1_s0.jpg", "p1_s1.jpg", "p3_s0.jpg", "p3_s1.jpg"]:
            shutil.copy(src / name, dst / name)
        parent_dirs.append(dst)
    meta_a = json.loads((SPEC_DIR / f"{a:03d}" / "meta.json").read_text())
    t0 = time.time()
    restore_vocab()
    cfg = TrainEvolutionConfig(
        initializer_token=meta_a["category"],
        parents_images_dirs=parent_dirs,
        placeholder_token=PLACEHOLDER,
        mix_weights=[0.5, 0.5],
        steps=200 if SMOKE_TEST else HYBRID_STEPS,
        seed=0,
        cache_root=Path(WEIGHTS),
        output_dir=d / "run",
    )
    coach = CoachEvolution(cfg)
    coach.train()
    emb = torch.load(d / "run" / "learned_embeds.bin")["t2"][PLACEHOLDER].float().cpu()
    torch.save(emb, d / "embedding.pt")
    meta = {"chapter": meta_a["chapter"], "category": meta_a["category"], "seed": 0,
            "negatives": [], "parents": [a, b], "minutes": round((time.time() - t0) / 60, 1)}
    (d / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=1))
    del coach
    restore_vocab()
    torch.cuda.empty_cache()
    make_plates(d)
    print(f"{d.name}：{meta['minutes']} 分")

In [ ]:
#@title 8. 座標を書き出し、図鑑用のデータにまとめる
restore_vocab()
F = torch.nn.functional
E = MODEL.clip_model.token_embedding.weight.detach().float()

# 英単語として完結しているトークン（"cat</w>" など）だけを「既知の語」とする
words = sorted((i, w[:-4]) for i, w in MODEL.tokenizer2.decoder.items()
               if i < E.shape[0] and re.fullmatch(r"[a-z]{3,}</w>", w))
EW = E[torch.tensor([i for i, _ in words], device=E.device)]
EWn = F.normalize(EW, dim=1)

# 既知の語の主成分が、文字体系の軸になる。各軸は語彙全体での標準偏差で割って揃える
K = 56
mu = EW.mean(0)
_, S, V = torch.pca_lowrank(EW - mu, q=64, center=False)
axes, scale = V[:, :K], S[:K] / (EW.shape[0] - 1) ** 0.5

def coords(e):
    return [round(v, 4) for v in (((e.to(E.device) - mu) @ axes) / scale).tolist()]

def nearest(e, k=12):
    sims = EWn @ F.normalize(e.to(E.device), dim=0)
    top = sims.topk(k)
    return [{"word": words[j][1], "cos": round(v, 4), "coords": coords(EW[j])}
            for v, j in zip(top.values.tolist(), top.indices.tolist())]

BOOK = Path(OUT) / ("book_data_smoke" if SMOKE_TEST else "book_data")
(BOOK / "images").mkdir(parents=True, exist_ok=True)

base = sorted(p for p in SPEC_DIR.iterdir() if re.fullmatch(r"\d{3}", p.name) and (p / "meta.json").exists())
hybrids = sorted(p for p in SPEC_DIR.iterdir() if p.name.startswith("h") and (p / "meta.json").exists())
next_id = 1 + max([json.loads((d / "meta.json").read_text())["id"] for d in base] or [0])
hybrid_ids = {d.name: next_id + k for k, d in enumerate(hybrids)}
specimens = []
for d in base + hybrids:
    meta = json.loads((d / "meta.json").read_text())
    sid = meta["id"] if "id" in meta else hybrid_ids[d.name]
    emb = torch.load(d / "embedding.pt")
    near = nearest(emb)
    images = []
    for k, name in enumerate(["p0_s0.jpg", "p1_s0.jpg", "p2_s0.jpg", "p3_s0.jpg"]):
        if (d / "plates" / name).exists():
            shutil.copy(d / "plates" / name, BOOK / "images" / f"{sid:03d}_{k}.jpg")
            images.append(f"images/{sid:03d}_{k}.jpg")
    specimens.append({
        "id": sid, "seed": meta["seed"], "chapter": meta["chapter"],
        "coords": coords(emb),
        "outlierness": round(1 - near[0]["cos"], 4),   # もっとも近い既知の語との距離（1 − cos）
        "negatives": meta["negatives"],
        "neighbors": near,
        "images": images,
        "parents": meta["parents"],
        "norm": round(emb.norm().item(), 4),
    })

# 種名の長さに使う「遠さ」は、本に収めた標本どうしの中で 0–1 に揃える
o = [s["outlierness"] for s in specimens]
lo, hi = min(o), max(o)
for s in specimens:
    s["farness"] = round((s["outlierness"] - lo) / (hi - lo), 4) if hi > lo else 0.5

rosetta = []
for w in ROSETTA_WORDS:
    i = MODEL.tokenizer2.encoder.get(w + "</w>")
    if i is not None:
        rosetta.append({"word": w, "coords": coords(E[i])})

data = {
    "mock": False,
    "meta": {
        "model": "Kandinsky 2.1・CLIP ViT-L/14 語彙空間",
        "created": time.strftime("%Y-%m-%d"),
        "steps": STEPS, "smoke_test": SMOKE_TEST,
        "vocabulary": f"英単語トークン {len(words)} 語の主成分 {K} 軸",
        "vlm": sorted({json.loads((d / "meta.json").read_text()).get("vlm", "BLIP-2 flan-t5-xl") for d in base}),
    },
    "chapters": [{"key": c["key"], "title": c["title"]} for c in CHAPTERS],
    "specimens": specimens,
    "rosetta": rosetta,
}
(BOOK / "specimens.js").write_text(
    "// 読めない博物誌 — colab/01_generate_specimens.ipynb が書き出したデータ\n"
    "window.HAKUBUTSUSHI = " + json.dumps(data, ensure_ascii=False) + ";\n")
archive = shutil.make_archive(str(BOOK), "zip", BOOK)
print(f"{len(specimens)} 体を書き出しました：{BOOK}")
for s in specimens:
    print(f"  No.{s['id']:03d}  遠さ {s['outlierness']:.3f}  近い語: {', '.join(w['word'] for w in s['neighbors'][:5])}")
print("zip:", archive)

## このあと

1. ドライブの `yomenai-hakubutsushi/book_data.zip`（試運転なら `book_data_smoke.zip`）を手元に落とし、中身を `book/data/` に置く
   - `book/data/specimens.js` と `book/data/images/` ができる形
2. `book/index.html` を開くと、試作データの代わりに本物の標本で図鑑が組まれる
3. 否定語は BLIP-2 の英語の答えのままです。和訳したい場合は、各標本に `"negatives_ja": ["猫", ...]` を足すと、そちらが使われます